# Notebook 1 — Method Overview: Representation Engineering for Lie Detection

## What this project is about

This project implements the **white-box lie-detection** methodology introduced in the paper
> *Representation Engineering: A Top-Down Approach to AI Transparency*  
> Zou et al., 2023 — and summarised in the LessWrong post *"Lie Detection for LLMs"*.

The central hypothesis is:

> **Language models internally encode whether a statement is true or false.  
> This signal is linearly readable from the model's hidden activations.**

If the hypothesis holds, we should be able to train a simple linear classifier — a **probe** — on those hidden vectors and use it to detect lies without any access to the model's weights or training data beyond the forward pass.

---

## 1. The RepE framework at a glance

### 1.1  Representation Engineering (RepE)

RepE treats the hidden states of a transformer as a *high-dimensional geometry* in which
human-interpretable concepts (truthfulness, sentiment, danger, …) correspond to
**linear directions**.

The pipeline has two stages:

| Stage | Goal | Output |
|-------|------|--------|
| **Reading** | Find the linear direction encoding a concept | A probe / control vector |
| **Writing** | Steer activations along that direction at inference time | Controlled model behaviour |

This notebook focuses on the **reading** stage — detecting truthfulness.  
The writing / steering stage is the basis for the ControlVector API in the RepEng GitHub repository.

### 1.2  Why hidden states?

Transformer models consist of stacked attention + MLP layers. After each layer $l$, every input
token $t$ is represented as a vector $\mathbf{h}^{(l)}_t \in \mathbb{R}^d$.

These hidden states carry increasingly abstract information as $l$ increases:
- **Early layers** — surface form, syntax, token identity
- **Middle layers** — semantic content, factual associations
- **Late layers** — task-specific reasoning, next-token probability

RepE extracts the hidden state at the **last token position** of a prompt, because by that
point the model has processed both the *question* and the *candidate answer*, and must
decide how to continue — the most informative moment.

---

## 2. Contrast pairs — the key data design

To find the *truthfulness direction*, RepE uses **contrast pairs**: two prompts that are
identical in structure but differ only in whether the answer is **true** or **false**.

```
Prompt A (true):
  Consider the correctness of the answer to the following question:
  Question: Which country contains the city Paris?
  Answer: France
  The probability of the answer being correct is

Prompt B (false):
  Consider the correctness of the answer to the following question:
  Question: Which country contains the city Paris?
  Answer: Italy
  The probability of the answer being correct is
```

The difference in hidden state between prompt A and prompt B isolates the truth signal.

**Grouped evaluation**: each *group* (= one question) has several candidate answers — exactly
one is correct. The probe is evaluated at the group level: it must rank the correct answer
above all incorrect ones. This is stricter than binary classification.

---

## 3. Probe training methods

### 3.1  Difference in Means (DIM)

The simplest method. The **truthfulness direction** is defined as:

$$\mathbf{d} = \frac{\mu^+ - \mu^-}{\|\mu^+ - \mu^-\|}$$

where $\mu^+$ is the centroid of all *true* hidden vectors and $\mu^-$ the centroid of all *false* ones.
A new vector $\mathbf{h}$ is classified as true if $(\mathbf{h} - \mathbf{c}) \cdot \mathbf{d} > 0$
where $\mathbf{c} = (\mu^+ + \mu^-)/2$.

Pros: closed-form, no fitting needed.  
Cons: ignores within-class variance.

### 3.2  Linear Algebraic Treatment (LAT) — the RepE paper's main method

LAT finds the truth direction without supervision labels on individual tokens.
It works on **random contrasts**:

1. Randomly pair activations without replacement.  
2. Compute the difference vector for each pair: $\mathbf{\delta}_i = \mathbf{h}_i^{(A)} - \mathbf{h}_i^{(B)}$.  
3. Centre those differences: $\tilde{\mathbf{\delta}}_i = \mathbf{\delta}_i - \bar{\mathbf{\delta}}$.  
4. Run PCA on $\{\tilde{\mathbf{\delta}}_i\}$ and take the **first principal component**.  

The first PC captures the direction of maximum variance in the contrast space — which, if
the hypothesis holds, is the direction along which truth/false pairs differ most consistently.

LAT is especially interesting because it can work **without labelled pairs**: it only needs a set
of prompts designed so that true and false statements differ systematically.

### 3.3  Logistic Regression (LR)

The standard supervised baseline: fit an L2-regularised logistic regression on the hidden
vectors. The decision boundary is a hyperplane, so LR is still a *linear* probe.

It typically achieves higher in-distribution accuracy than DIM or LAT, but its generalisation
to new datasets can be weaker if it overfits to dataset-specific features.

### 3.4  Grouped PCA (PCA-G)

A variant of LAT that applies within-group centering before PCA:

1. For each question group $g$, subtract the group mean from all members.  
2. Stack all centred vectors and run PCA.  

This removes between-group variation (e.g. different topics) so PCA focuses purely on the
within-group contrast — the truth/false difference.

---

## 4. Generalisation — the key research question

Training a probe that works in-distribution is relatively easy. The harder and more interesting
question is:

> **If a probe is trained on geography questions, does it still detect lies in arithmetic questions?**

A truly **universal truth direction** would transfer across domains. That is the central
claim of the RepE paper, and it is what Notebooks 3 and 5 test systematically.

---

## 5. Datasets used in this project

| Dataset | Task type | Groups | Candidates per group |
|---------|-----------|--------|---------------------|
| `cities` | Geography: which country is a capital in? | 10 | 4 (1 correct) |
| `larger_than` | Arithmetic: is A > B? | 10 | 2 (yes / no) |
| `qa` | General knowledge short-answer | 8 | 4 (1 correct) |
| `repeng_truthful` | Self-report honesty statements from RepEng repo | up to 40 | 2 (honest / dishonest) |

All prompts use the same template so the only variable is the factual content.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from lie_detector_llm.datasets import build_dataset_collection

collection = build_dataset_collection(include_repeng_truthful=True)
collection.frame.head(10)

## Dataset statistics

In [ ]:
collection.frame.groupby('dataset_name').agg(
    n_rows=('dataset_name', 'size'),
    n_groups=('group_id', 'nunique'),
    n_true=('label', 'sum'),
)

## The prompt template

This is the exact framing sent to the model. The model never actually answers — we collect
hidden states *just before* it would generate the next token.

In [ ]:
# Show one full prompt for each dataset
for dataset_name in collection.dataset_names():
    example = collection.subset(dataset_name).iloc[0]
    print(f"=== {dataset_name} ===")
    print(example['prompt'])
    print(f"  label = {example['label']}\n")

## Grouped evaluation explained

Within each group, the probe assigns a score to every candidate.
We pick the candidate with the **highest score** as the model's prediction.
Grouped accuracy = fraction of groups where the prediction is the correct answer.

This is harder than binary classification: even if the probe correctly labels 3 out of 4
candidates, it still fails the group if it assigns the highest score to a wrong candidate.

In [ ]:
# Show one complete group — cities question with 4 candidates
cities_frame = collection.subset('cities')
first_group = cities_frame['group_id'].iloc[0]
cities_frame[cities_frame['group_id'] == first_group][['question', 'answer', 'label', 'prompt']].reset_index(drop=True)

## What the hidden states look like

For a model with $L$ transformer layers and hidden dimension $d$:
- Each prompt → a tensor of shape $(L, d)$ (one vector per layer, taken at the last token)
- Once we pick one layer → each prompt is a point in $\mathbb{R}^d$

For `distilgpt2`: $L = 6$, $d = 768$.  

The probe operates in this 768-dimensional space and finds a hyperplane that separates
true from false hidden vectors. Notebooks 2 and 4 show which layers give the best separation.